In [ ]:
!pip -q install playwright pandas
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 32.7 MB/s eta 0:00:00
191.1 MiB [] 0% 0.0s191.1 MiB [] 0% 257.7s191.1 MiB [] 0% 269.5s191.1 MiB [] 0% 816.0s191.1 MiB [] 0% 679.2s191.1 MiB [] 0% 946.9s191.1 MiB [] 0% 825.6s191.1 MiB [] 0% 740.7s191.1 MiB [] 0% 681.7s191.1 MiB [] 0% 646.6s191.1 MiB [] 0% 614.8s191.1 MiB [] 0% 588.9s191.1 MiB [] 0% 560.1s191.1 MiB [] 0% 538.6s191.1 MiB [] 0% 521.0s191.1 MiB [] 0% 504.1s191.1 MiB [] 0% 488.6s191.1 MiB [] 0% 480.7s191.1 MiB [] 0% 436.6s191.1 MiB [] 0% 415.1s191.1 MiB [] 0% 387.5s191.1 MiB [] 0% 365.2s191.1 MiB [] 0% 346.2s191.1 MiB [] 0% 335.3s191.1 MiB [] 0% 310.3s191.1 MiB [] 0% 299.2s191.1 MiB [] 0% 288.9s191.1 MiB [] 0% 278.1s191.1 MiB [] 0% 257.7s191.1 MiB [] 0% 245.3s191.1 MiB [] 0% 234.2s191.1 MiB [] 0% 224.2s191.1 MiB [] 0% 214.7s191.1 MiB [] 0% 201.1s191.1 MiB [] 0% 188.4s191.1 MiB [] 0% 179.0s191.1 MiB [] 0% 169.4s191.1 MiB [] 0% 161.3s191.1 

In [ ]:
!python -m playwright install --with-deps chromium

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,549 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,080 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubun

In [ ]:

from playwright.async_api import async_playwright

async def test_browser():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )

        page = await browser.new_page()
        await page.goto("https://example.com")

        print("瀏覽器啟動成功")
        print("網頁標題:", await page.title())

        await browser.close()

await test_browser()


瀏覽器啟動成功
網頁標題: Example Domain


In [ ]:

import asyncio
import json
from playwright.async_api import async_playwright

STOCK_ID = "2330"

URL = (
    f"https://www.cmoney.tw/forum/stock/"
    f"{STOCK_ID}?tab=AISummary"
)

async def scrape_cmoney(stock_id):
    results = {}
    api_status = {}

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )

        page = await browser.new_page(
            locale="zh-TW",
            viewport={"width": 1440, "height": 900}
        )

        async def capture_response(response):
            for api_name in ["AiSentiment", "AiSummary"]:
                target = f"/{api_name}/Stock/{stock_id}"

                if target in response.url:
                    api_status[api_name] = response.status

                    print(
                        f"{api_name} HTTP:",
                        response.status
                    )

                    if response.status == 200:
                        try:
                            results[api_name] = await response.json()
                            print(f"{api_name} JSON 取得成功")
                        except Exception as e:
                            print("JSON 解析失敗:", repr(e))

        pending_tasks = []

        def on_response(response):
            task = asyncio.create_task(
                capture_response(response)
            )
            pending_tasks.append(task)

        page.on("response", on_response)

        try:
            response = await page.goto(
                URL,
                wait_until="domcontentloaded",
                timeout=45000
            )

            print(
                "頁面 HTTP:",
                response.status if response else None
            )

            # 最多等待 20 秒，觀察兩個 API
            for _ in range(20):
                if len(results) == 2:
                    break
                await page.wait_for_timeout(1000)

            if pending_tasks:
                await asyncio.gather(
                    *pending_tasks,
                    return_exceptions=True
                )

        except Exception as e:
            print("發生錯誤:", repr(e))

        finally:
            await browser.close()

    print("\n===== 擷取結果 =====")
    print("API 狀態:", api_status)
    print("成功取得:", list(results.keys()))

    for name, data in results.items():
        filename = f"{stock_id}_{name}.json"

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(
                data,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"\n{name}:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:1200])
        print("已儲存:", filename)

    return results

results = await scrape_cmoney(STOCK_ID)


頁面 HTTP: 200
AiSentiment HTTP: 200
AiSummary HTTP: 200
AiSentiment JSON 取得成功
AiSummary JSON 取得成功
AiSentiment HTTP: 200
AiSummary HTTP: 200
AiSentiment JSON 取得成功
AiSummary JSON 取得成功

===== 擷取結果 =====
API 狀態: {'AiSentiment': 200, 'AiSummary': 200}
成功取得: ['AiSentiment', 'AiSummary']

AiSentiment:
{
  "sentimentScore": 40,
  "sampleCount": 158,
  "state": "normal",
  "distribution": {
    "strongBull": 6,
    "bull": 13,
    "neutral": 71,
    "bear": 50,
    "strongBear": 18
  },
  "startTime": 1792623600000,
  "endTime": 1792648800000
}
已儲存: 2330_AiSentiment.json

AiSummary:
{
  "endTimeGroups": [
    {
      "endTime": 1792648800000,
      "summaries": [
        {
          "startTime": 1792623600000,
          "endTime": 1792648800000,
          "title": "外資動向與融資槓桿高漲的拉扯現象",
          "description": "市場焦點轉向外資動向與融資槓桿壓力，出現「外資亂賣」等留言，讓人擔心指數走勢。也有同學指出「殺融資，大家都跑去買進金融股避險」的現象，這股風向改變讓人更關注權值與金融股的表現。",
          "articleIds": [
            186513304,
            186511442
          ],
          "sen

In [ ]:
import json

with open("2330_AiSummary.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "endTimeGroups": [
    {
      "endTime": 1792648800000,
      "summaries": [
        {
          "startTime": 1792623600000,
          "endTime": 1792648800000,
          "title": "外資動向與融資槓桿高漲的拉扯現象",
          "description": "市場焦點轉向外資動向與融資槓桿壓力，出現「外資亂賣」等留言，讓人擔心指數走勢。也有同學指出「殺融資，大家都跑去買進金融股避險」的現象，這股風向改變讓人更關注權值與金融股的表現。",
          "articleIds": [
            186513304,
            186511442
          ],
          "sentiment": -1
        },
        {
          "startTime": 1792623600000,
          "endTime": 1792648800000,
          "title": "同學在吵台積電成為市場核心動力",
          "description": "群內討論圍繞台積電與盤勢走向，焦點在2400附近的支撐與破位風險。有人喊「今日必破2400，殺爆融資」。也有人警告別亂空亂砍：「不要跟著亂空亂砍……」。整體氛圍偏看多，但對權值股與融資槓桿的變化仍在討論，台積電的動向被視為市場走勢的關鍵。",
          "articleIds": [
            186503253,
            186503521
          ],
          "sentiment": 0
        }
      ],
      "sourceCount": 191,
      "sentiment": {
        "sentimentScore": 40
      }
    },
    {
      "endTime": 1792623600000,
      "summaries": [
        {

In [ ]:
with open("2330_AiSentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "sentimentScore": 40,
  "sampleCount": 158,
  "state": "normal",
  "distribution": {
    "strongBull": 6,
    "bull": 13,
    "neutral": 71,
    "bear": 50,
    "strongBear": 18
  },
  "startTime": 1792623600000,
  "endTime": 1792648800000
}
